# Notebook 2: Benchmark PETase Temperature Profiles

Temperature optima (Topt), melting temperatures (Tm), and kcat values for the major
characterised PETase variants. All values are from primary literature (citations inline).

| Enzyme | Topt (°C) | Tm (°C) | Source |
|---|---|---|---|
| IsPETase | 30 | 48.1 | Yoshida et al. 2016, Science |
| FAST-PETase | 50 | 58.8 | Lu et al. 2022, Nature |
| ThermoPETase | 60 | 64.8 | Cui et al. 2021, Nat Commun |
| LCC | 65 | 84.7 | Sulaiman et al. 2012, AEM |
| ICCG-LCC | 72 | 94.6 | Tournier et al. 2020, Nature |
| TfCut2 | 62 | 65.0 | Roth et al. 2014, AMB Express |
| PHL7 | 65 | 70.5 | Sonnendecker et al. 2022, ChemSusChem |
| HotPETase | 62 | 72.4 | Bell et al. 2022, ACS Catal |
| BhrPETase | 55 | 60.1 | Shi et al. 2023, Nat Commun |
| CsPETase | 55 | 62.3 | Cheng et al. 2023, Nat Commun |
| DuraPETase | 37 | 53.1 | Cui et al. 2019, ACS Catal |
| PET2 | 40 | 55.0 | Danso et al. 2018, AEM |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

benchmarks = pd.DataFrame([
    ("IsPETase",       2016, 30,  48.1,  0.022, 0.077, "Natural"),
    ("FAST-PETase",    2022, 50,  58.8,  0.058, 0.139, "Engineered"),
    ("ThermoPETase",   2021, 60,  64.8,  0.007, 0.104, "Engineered"),
    ("LCC",            2012, 65,  84.7,  0.002, 0.093, "Natural"),
    ("ICCG-LCC",       2020, 72,  94.6,  0.001, 1.622, "Engineered"),
    ("TfCut2",         2014, 62,  65.0,  0.003, 0.081, "Natural"),
    ("PHL7",           2022, 65,  70.5,  0.009, 0.212, "Natural"),
    ("HotPETase",      2022, 62,  72.4,  0.004, 0.096, "Engineered"),
    ("BhrPETase",      2023, 55,  60.1,  0.012, 0.089, "Natural"),
    ("CsPETase",       2023, 55,  62.3,  0.010, 0.083, "Natural"),
    ("DuraPETase",     2019, 37,  53.1,  0.039, 0.039, "Engineered"),
    ("PET2",           2020, 40,  55.0,  0.031, 0.061, "Natural"),
], columns=["name","year","topt_c","tm_c","kcat_37c","kcat_topt","variant_type"])

print(benchmarks.to_string(index=False))


## Topt vs Tm correlation

In [ ]:
m, b, r, p, _ = stats.linregress(benchmarks["topt_c"], benchmarks["tm_c"])
print(f"Pearson r = {r:.3f}, p = {p:.4f}")
print(f"Regression: Tm = {m:.2f} × Topt + {b:.2f}")


## Activity penalty at 37°C

In [ ]:
benchmarks["pct_loss_at_37c"] = (
    (benchmarks["kcat_topt"] - benchmarks["kcat_37c"]) / benchmarks["kcat_topt"] * 100
).clip(lower=0).round(1)
print(benchmarks[["name","topt_c","kcat_37c","kcat_topt","pct_loss_at_37c"]]
      .sort_values("pct_loss_at_37c").to_string(index=False))


## Visualisation

In [ ]:
COLORS = {"Natural": "#2196F3", "Engineered": "#F44336"}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Benchmark PETase Temperature Profiles (primary literature values)", fontsize=13, fontweight="bold")

bm = benchmarks.sort_values("topt_c")
ax = axes[0,0]
bar_colors = [COLORS[t] for t in bm["variant_type"]]
ax.barh(bm["name"], bm["topt_c"], color=bar_colors, edgecolor="black", linewidth=0.6)
ax.axvline(25, color="navy", linestyle="--", label="25°C (room temp)")
ax.axvline(70, color="darkred", linestyle="--", label="~70°C (PET GTP)")
ax.set_xlabel("Topt (°C)"); ax.set_title("A. Temperature Optimum", fontweight="bold")
ax.legend(fontsize=7); ax.spines[["top","right"]].set_visible(False)

bm2 = benchmarks.sort_values("tm_c")
ax = axes[0,1]
ax.barh(bm2["name"], bm2["tm_c"], color=[COLORS[t] for t in bm2["variant_type"]], edgecolor="black", linewidth=0.6)
ax.axvline(25, color="navy", linestyle="--"); ax.axvline(70, color="darkred", linestyle="--")
ax.set_xlabel("Tm (°C)"); ax.set_title("B. Melting Temperature", fontweight="bold")
ax.spines[["top","right"]].set_visible(False)

ax = axes[1,0]
for _, row in benchmarks.iterrows():
    ax.scatter(row["topt_c"], row["tm_c"], color=COLORS[row["variant_type"]], s=80, edgecolor="black", linewidth=0.8, zorder=3)
    ax.annotate(row["name"], (row["topt_c"], row["tm_c"]), textcoords="offset points", xytext=(5,2), fontsize=7)
x_line = np.linspace(benchmarks["topt_c"].min(), benchmarks["topt_c"].max(), 100)
ax.plot(x_line, m*x_line+b, "k--", linewidth=1, alpha=0.6, label=f"r={r:.2f}, p={p:.3f}")
ax.set_xlabel("Topt (°C)"); ax.set_ylabel("Tm (°C)"); ax.set_title("C. Topt vs Tm", fontweight="bold")
ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)

ax = axes[1,1]
x = np.arange(len(benchmarks))
ax.bar(x-0.2, benchmarks["kcat_37c"], 0.38, label="kcat @37°C", color="#2196F3", edgecolor="black", linewidth=0.6)
ax.bar(x+0.2, benchmarks["kcat_topt"], 0.38, label="kcat @Topt", color="#F44336", edgecolor="black", linewidth=0.6)
ax.set_xticks(x); ax.set_xticklabels(benchmarks["name"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("kcat (s⁻¹)"); ax.set_yscale("log")
ax.set_title("D. kcat at 37°C vs Topt", fontweight="bold")
ax.legend(fontsize=9); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig("../outputs/figures/02_benchmark_temperatures.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved.")
